In [1]:
#import logging
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
import torch.optim as optim
#from torch.utils.tensorboard import SummaryWriter
import torch
import unicodedata
import string
from tqdm import tqdm
import random

from typing import List
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from typing import Any, Dict
from pathlib import Path
import math
import h5py
import time
import re
import argparse
import timm
#from torch.utils.tensorboard import SummaryWriter

from aroma_Burgers import *
from DiT_Burgers import *
from metrics_Burgers import *

In [6]:
A=torch.randn(3)
B=A.expand(2,3)
B[1:,:].shape

torch.Size([1, 3])

In [9]:

#fname=Path("/Users/adogbo/Documents/ML/Cours/Projet/AROMA_PREF/Simu2D/datasets/rayleigh_benard/data/test/rayleigh_benard_Rayleigh_1e9_Prandtl_5.hdf5")
fname=Path("/Users/adogbo/Documents/ML/Cours/Projet/AROMA_PREF/Simu2D/datasets/rayleigh_benard/data/test/rayleigh_benard_Rayleigh_1e9_Prandtl_2e-1.hdf5")
with h5py.File(fname, "r") as f:
        print(f.metadata.field_names)
        #data = f[][:] #shape (B, T, N )

OSError: Unable to synchronously open file (truncated file: eof = 285605888, sblock->base_addr = 0, stored_eof = 1048600256)

## Test

In [ ]:
# For checkpoint
direc="/Users/adogbo/Documents/ML/Cours/Projet/AROMA_PREF/checkpoints"
#direc = Path(cfg.save_dir)
#save_dir.mkdir(parents=True, exist_ok=True)
#ckpt_path = save_dir / "checkpoint.pt"



checkpoint_los_tr=Path(direc+"/losses_train.npy")
checkpoint_los_te=Path(direc+"/losses_test.npy")

checkpooint_losTr_encodecoDPO=Path(direc+"/losses_train_enco_deco_DPO.npy")
checkpooint_losTe_encodecoDPO=Path(direc+"/losses_test_enco_deco_DPO.npy")

"""checkpoint_DiT = Path(direc+"/checkpoint_DiT.pt")
checkpoint_los_tr_DiT=Path(direc+"/losses_train_DiT.npy")
checkpoint_los_te_DiT=Path(direc+"/losses_test_DiT.npy")
checkpoint_los_te_latent_DiT=Path(direc+"/losses_test_latent_DiT.npy")
"""


loss_train_encodeco=np.load(checkpoint_los_tr)
loss_test_encodeco=np.load(checkpoint_los_te)

loss_train_encodecoDPO=np.load(checkpooint_losTr_encodecoDPO)
loss_test_encodecoDPO=np.load(checkpooint_losTe_encodecoDPO)





### CHECK NAN

In [ ]:
### CHECK LES PArametres qui ont nan
print("Nombre de parametres nan =",len(state_nan.name_param_nan))
print("Nombre de parametres avec grad nan =",len(state_nan.name_param_gradNaN))

print("Nombre total de parametres=",len( [name for name,_ in list(state_nan.model_enco.named_parameters()) +list(state_nan.model_deco.named_parameters())] ))

print("y'a t il nan dans les coordonne?:",torch.isnan(state_nan.x).any())
print("y'a t il nan dans les velocity?:",torch.isnan(state_nan.u).any())


In [ ]:
### CHECK NAN in sample

if torch.isnan(state_nan.means).any():
    print("nan in means")
    indices_nan_means = torch.where(torch.isnan(state_nan.means))
    print("Indices des NaN dans means:", indices_nan_means)  # tensor([1, 3])

if torch.isnan(state_nan.log_var_square).any():
    print("nan in means")
    indices_nan_log_var_square = torch.where(torch.isnan(state_nan.log_var_square))
    print("Indices des NaN dans log_var_square:",  indices_nan_log_var_square)  # tensor([1, 3])

if torch.isnan(state_nan.sample).any():
    print("nan in means")
    indices_nan_sample = torch.where(torch.isnan(state_nan.sample))
    print("Indices des NaN dans sample:",  indices_nan_sample)  # tensor([1, 3])
len(indices_nan_sample[0])

In [ ]:
#### CHECK NAN IN THE ENCODER
for name, param in state_nan.model_enco.named_parameters():
    if torch.isnan(param).any():
        print("NaN dans les poids :", name)
    

In [ ]:
##### CHECK NAN IN THE DECODER
for name, param in state_nan.model_deco.named_parameters():
    if torch.isnan(param).any():
        print("NaN dans les poids :", name)

In [ ]:
#### CHECK NAN IN THE grad of ENCODER
for name, param in state_nan.model_enco.named_parameters():
    if param.grad is not None:
        if torch.isnan(param.grad).any():
            print("NaN dans le gradient :", name)
    
    else:
        print(f"{name} has None gradient")


In [ ]:
## Calcule the model from Nan
u_nan=rearrange(state_nan.u, "b T N d -> (b T) N d")

sample,means,log_var_square,_=state_nan.model_enco(state_nan.x,u_nan)

if torch.isnan(sample).any():
    print("nan in means")
    indices_nan_means = torch.where(torch.isnan(sample))
    print("Indices des NaN dans means:", indices_nan_means)  # tensor([1, 3])


In [ ]:
class myModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear=nn.Linear(10,2)

    def forward(self,x):
        return self.linear(x)

mymodel=myModel()

for name,para in mymodel.named_parameters():
    if para.grad is  None:
        print(name, para.grad)


### DISPLAY LOSS

In [ ]:
## Caracteristiques of state
print("epoche of enco deco train",state_enco_deco.epoch)
print("loss_train enco decp shape",loss_train.shape)
print("loss_test  enco deco shape",loss_test.shape)

print("epoche of DiT train",state_DiT.epoch)
print("loss_train DiTshape",loss_train_DiT.shape)
print("loss_test  DiT shape",loss_test_DiT.shape)
print("loss_latent_test  DiT shape",loss_test_latent_DiT.shape)

#print(loss_test_latent_DiT)


### Plot losses

In [ ]:
# Plot à différents instants
plt.figure(figsize=(12, 6))

# Sélectionner quelques instants
time_indices = [0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, 2))

#for idx, color in zip(time_indices, colors):
plt.plot(loss_train_encodeco, 
            color=colors[0], 
            label=f'train eoc_deco',
            linewidth=2)

plt.plot(loss_test_encodeco, 
            color=colors[1], 
            label=f'test enco_deco',
            linewidth=2)



plt.xlabel('epoch', fontsize=12)
plt.ylabel('losses', fontsize=12)
plt.title('Losses encoder decoder train/test', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Plot à différents instants
plt.figure(figsize=(12, 6))

# Sélectionner quelques instants
time_indices = [0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, 3))

#for idx, color in zip(time_indices, colors):
plt.plot( loss_train_encodecoDPO, 
            color=colors[0], 
            label=f'train_enco_deco_DPO',
            linewidth=2)

plt.plot(loss_test_encodecoDPO, 
            color=colors[1], 
            label=f'test_enco_deco_DPO',
            linewidth=2)



plt.xlabel('epoch', fontsize=12)
plt.ylabel('losses', fontsize=12)
plt.title('Losses enco_decoDPO train/test', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Here we are testing the model

### TEST of encpder decoder

In [ ]:

checkpoint_f = Path(direc+"/checkpoint.pt")

checkpoint_DPO = Path(direc+"/checkpoint_enco_deco_DPO.pt")

checkpoint_DPOlast = Path(direc+"/checkpoint_enco_deco_DPO_lastModel.pt")

print("resolve==============",checkpoint_f.resolve())


if checkpoint_f.exists() and checkpoint_f.is_file():
    with checkpoint_f.open("rb") as fp:
        state_enco_deco=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state enco deco reussi----------------------")

if checkpoint_DPO.exists() and checkpoint_DPO.is_file():
    with checkpoint_DPO.open("rb") as fp:
        state_enco_deco_DPO=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state DPO reussi----------------------")

if checkpoint_DPOlast.exists() and checkpoint_DPOlast.is_file():
    with checkpoint_DPOlast.open("rb") as fp:
        state_enco_deco_DPOl=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
        
        
        print("-------------------telechargemnt state DPO last reussi----------------------")

        
    """with checkpoint_nan.open("rb") as fp: 
        state_nan=torch.load(fp,weights_only=False,map_location=torch.device('cpu'))
        print("-------------------telechargemnt state_nan reussi----------------------")"""

In [ ]:
base = Path("/Users/adogbo/Documents/ML/Cours/Projet/AROMA_PREF")
fname = base / "dataset" /"CE_test_E1.h5"
with h5py.File(fname, "r") as f:
    data = f["test/pde_250-100"][:]

x_dim=1
batch_size=4
test_data=Dataset_pde(data,x_dim,x_min=0,x_max=16,t_min=0,t_max=4,forescast=False)

#kwargs = {"num_workers": 4, "pin_memory": True} if cfg.use_gpu else {}
test_loader=DataLoader(dataset=test_data, batch_size=batch_size, shuffle=False)
test_loader_cycle=cycle(test_loader)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
device=torch.device("cpu")


iter=1

beta=0.0001

with torch.no_grad():
    for itr in range(iter):
        batch=next(test_loader_cycle)
        state_enco_deco.model_enco.eval()
        state_enco_deco.model_deco.eval()

        state_enco_deco_DPO.model_enco.eval()
        state_enco_deco_DPO.model_deco.eval()
        x=batch[0]
        u=batch[1]

        x=x.to(device)
        u=u.to(device)
        target=u
    
        batch_size_test,time_step,space_step,_=x.shape #u.shape
        
        x=rearrange(x, "b T N d -> (b T) N d")
        u=rearrange(u, "b T N d -> (b T) N d")

        sample,_,_,_=state_enco_deco.model_enco(x,u)
        out=state_enco_deco.model_deco(sample,x)
        out0=rearrange(out,"(b T) N d -> b T N d",b=batch_size_test)

        sample,_,_,_=state_enco_deco_DPO.model_enco(x,u)
        out=state_enco_deco_DPO.model_deco(sample,x)
        out_DPO=rearrange(out,"(b T) N d -> b T N d",b=batch_size_test)

        sample,_,_,_=state_enco_deco_DPOl.model_enco(x,u)
        out=state_enco_deco_DPOl.model_deco(sample,x)
        out_DPOl=rearrange(out,"(b T) N d -> b T N d",b=batch_size_test)
        


        

In [ ]:
traj_no=3
time_no=0
nx=out0.shape[2]

solution_approx=out0[traj_no,time_no] 
solution_approxDPO=out_DPO[traj_no,time_no] 
solution_approxDPOl=out_DPOl[traj_no,time_no] 

solution_true=target[traj_no,time_no]

sol_appr=solution_approx.detach().cpu().numpy()
sol_apprDPO=solution_approxDPO.detach().cpu().numpy()
sol_apprDPOl=solution_approxDPOl.detach().cpu().numpy()
sol_true=solution_true.detach().cpu().numpy()
x = np.linspace(0, 16, nx)
#t = np.linspace(0, 4, nt,endpoint=True)

# Plot à différents instants
plt.figure(figsize=(12, 6))

# Sélectionner quelques instants
time_indices = [0, 1,2,3]
colors = plt.cm.viridis(np.linspace(0, 1, len(time_indices)))

#for idx, color in zip(time_indices, colors):

plt.plot(x, sol_appr, label=f'approx',linewidth=2)

#plt.plot(x, sol_apprDPO, label=f'approxDPO',linewidth=2)

plt.plot(x, sol_apprDPOl, 
            label=f'approxDPOl',
            linewidth=2)

plt.plot(x, sol_true, 
            label=f'true',
            linewidth=2)

plt.xlabel('x', fontsize=12)
plt.ylabel('u(x,t)', fontsize=12)
plt.title('Solution de l\'équation de Burgers', fontsize=14)

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
mass_weight=10.
energy_weight=0.1
grad_weight=30.
boundary_weight=0.1
dx=0.16
phy_metric0=physicalMetricsBurgers(out0.unsqueeze(0),target.unsqueeze(0),dx,mass_weight,energy_weight,grad_weight,boundary_weight)
phy_metric_DPOl=physicalMetricsBurgers(out_DPOl.unsqueeze(0),target.unsqueeze(0),dx,mass_weight,energy_weight,grad_weight,boundary_weight)

In [ ]:
plt.plot(phy_metric0[0,3], label=f'approx',linewidth=2)
plt.plot(phy_metric_DPOl[0,3], label=f'DPO',linewidth=2)

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
#print(phy_metric0.shape)

In [ ]:
masss=torch.sum(target[0,:,:,0],dim=1)
#masss

### TEST with Encoder DIT Decoder

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
device=torch.device("cpu")
test_losses=[]
iter=10

beta=0.0001

min_noise_std = 1e-2 #cfg.min_noise_std  # 2e-6
num_refinement_steps=3 #cfg.num_refinement_steps
betas = [
    min_noise_std ** (k / num_refinement_steps)
    for k in reversed(range(num_refinement_steps + 1))
]

scheduler = DDPMScheduler(
    num_train_timesteps=num_refinement_steps + 1,
    trained_betas=betas,
    prediction_type="v_prediction",
    clip_sample=False,
)
time_multiplier = 1000 / num_refinement_steps

test_losses_latent=[]
test_losses=[]
with torch.no_grad():
    for itr in range(iter):
        batch=next(test_loader_cycle)
        state_enco_deco.model_enco.eval()
        state_enco_deco.model_deco.eval()
        state_DiT.model.eval()
        #state_nan.model_enco.eval()
        #state_nan.model_deco.eval()
        x=batch[0]
        
        x=batch[0]
        u=batch[1]
        x=x.to(device)
        u=u.to(device)
        target=u

        batch_size_test,time_step_test,space_step,_=x.shape #u.shape
        x=rearrange(x, "b T N d -> (b T) N d")
        u=rearrange(u, "b T N d -> (b T) N d")


        sample,means,log_var_square,_=state_enco_deco.model_enco(x,u)

        sammple_amenaged=rearrange(sample,"(b T) N d -> b T N d",b=batch_size_test)
        total_test_loss_per_time=0 

        y=[]  
        for t in range(time_step_test - 1 ):
            
            sample_prev=sammple_amenaged[:,t,:,:]  #sample_prev shape (b,N,d)
            sample_cur=sammple_amenaged[:,t+1,:,:] #sample_cur
            y_noised = torch.randn_like(sample_cur)  # , dtype=sample_cur.dtype, device=sample_cur.device


            for k in scheduler.timesteps:
                timess = (
                    torch.zeros(
                        size=(sample_cur.shape[0],), dtype=sample_cur.dtype, device=sample_cur.device
                    )
                    + k
                )
                pred = state_DiT.model(
                    torch.cat([sample_prev, y_noised], dim=1), timess * time_multiplier
                )
                y_noised = scheduler.step(pred, k, y_noised).prev_sample # shape (b,M,h)
                
            loss_test=F.mse_loss(y_noised,sample_cur)
            total_test_loss_per_time+=loss_test/(time_step_test - 1 )
            y.append( y_noised.unsqueeze(1)) 
            #print("len de y", len(y))

        test_losses_latent.append(total_test_loss_per_time.cpu().numpy())
        print(f"------------ itr={itr} test_losses_latent= {total_test_loss_per_time.cpu().numpy()}")
        
        y=torch.cat(y,dim=1)  # # shape (b,T-1,M,h)
        #print("y before cat ",y.shape)
        y=rearrange(y, "b T M h -> (b T) M h")

        x_out=rearrange(x, "(b T) M h -> b T M h",b=batch_size_test)
        x_out=rearrange(x_out[:,1:,:,:], "b T M h -> (b T) M h")
        #print("y.shape",y.shape)
        #print("x_out.shape",x_out.shape)
        out=state_enco_deco.model_deco(y,x_out) # shape (b,T-1,N,d)
        out=rearrange(out, "(b T) M h -> b T M h",b=batch_size_test)
        loss_gen= F.mse_loss(target[:,1:,:,:],out)
        print(f"itr={itr} test_losses= {loss_gen.cpu().numpy()}")
        test_losses.append(loss_gen.cpu().numpy())

        

    # Supposons vos données
    # solution.shape = (nt, nx)
    # nt = nombre de pas de temps
    # nx = nombre de points spatiaux

    # Création des axes
    
    

In [ ]:
traj_no=0
time_no=240
nx=out.shape[2]
solution_approx=out[traj_no,time_no] 
solution_true=target[traj_no,time_no+1]
sol_appr=solution_approx.detach().cpu().numpy()
sol_true=solution_true.detach().cpu().numpy()
x = np.linspace(0, 16, nx)
#t = np.linspace(0, 4, nt,endpoint=True)

# Plot à différents instants
plt.figure(figsize=(12, 6))

# Sélectionner quelques instants
time_indices = [0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, len(time_indices)))

#for idx, color in zip(time_indices, colors):
plt.plot(x, sol_appr, 
            color=colors[0], 
            label=f'approx',
            linewidth=2)

plt.plot(x, sol_true, 
            color=colors[1], 
            label=f'true',
            linewidth=2)

plt.xlabel('x', fontsize=12)
plt.ylabel('u(x,t)', fontsize=12)
plt.title('Solution de l\'équation de Burgers', fontsize=14)

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### TEST DDPM

In [ ]:
#sample[2]
#sample,means,log_var_square
from diffusers.schedulers import DDPMScheduler

num_refinement_steps=3
min_noise_std=2e-6
betas = [
    min_noise_std ** (k / num_refinement_steps)
    for k in reversed(range(num_refinement_steps + 1))
]
scheduler = DDPMScheduler(
        num_train_timesteps=3+1,
        trained_betas=betas,
        prediction_type="v_prediction",
        clip_sample=False,
    )
inputr=torch.randn(10,5,2)    
k = torch.randint(
                        0,
                        scheduler.config.num_train_timesteps,
                        (inputr.shape[0],),
                        device=inputr.device,
                    )
noise_factor = scheduler.alphas_cumprod.to(inputr.device)[k]
print("noise_factor.shape",noise_factor.shape)
noise_factor = noise_factor.view(-1, *[1 for _ in range(inputr.ndim - 1)])
signal_factor = 1 - noise_factor
print("AFTER reshape noise_factor.shape",noise_factor.shape)

print(scheduler.alphas_cumprod)



## Bruoillon

In [ ]:
x_dim=1
u_dim=1
N=100
batch_size=4
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)
#device= torch.device("mps")
#device= torch.device("cpu")

x=torch.rand(batch_size,N,x_dim).to(device)
u=torch.randn(batch_size,N,u_dim).to(device)
beta=1
M=10
d=64
h=8
log_scale_min=0
log_scale_max=0
k=16
num_enco_head=8
attn_dropout=0
enco_dropout=0
mult_dim_ff=4
use_gelu=True

num_self_attn_deco=3
out_dim=u_dim
mult_dim_deco=4
hidden_dim_deco=128

print("-------------- Loading train data-------------")
base = Path("/Users/adogbo/Documents/ML/Cours/Projet/AROMA_PREF")
fname = base / "dataset" /"CE_train_E1.h5"
with h5py.File(fname, "r") as f:
    data_tr = f["train/pde_250-100"][:]

train_data=Dataset_pde(data_tr,x_dim,x_min=0,x_max=16,t_min=0,t_max=4,forescast=False)

"""print("-------------- Loading test data-------------")
fname="CE_test_E1.h5"
with h5py.File(fname, "r") as f:
    data_te = f["test/pde_250-100"][:]"""

#test_data=Dataset_pde(data_te,x_dim,x_min=0,x_max=16,t_min=0,t_max=4,forescast=False)

kwargs = {"num_workers": 4, "pin_memory": True} if False else {}
train_loader=DataLoader(dataset=train_data, batch_size=batch_size, shuffle=False,**kwargs)
#test_loader=DataLoader(dataset=test_data, batch_size=batch_size, shuffle=False,**kwargs)
print(f"shape of data train: {data_tr.shape}")
print(f"The len of dataset of train :{len(train_data)}")
print(f"the len of dataloader train:{len(train_loader)}")

In [ ]:
enco=Encoder(x_dim,u_dim,M,d,h,log_scale_min,log_scale_max,k,device,num_enco_head,attn_dropout=attn_dropout,enco_dropout=enco_dropout,
            mult_dim_ff=mult_dim_ff, use_pi=True,log_sampling=True,include_input=True,use_gelu=use_gelu,enco_geo=True,include_pos_in_value=False,fourrier_feature_type="base2")
enco.to(device=device)
enco=enco.to(dtype=torch.float32)

deco=Decoder(h,d,k,num_self_attn_deco,mult_dim_deco,x_dim,out_dim,hidden_dim_deco,log_scale_min,log_scale_max,device,
                use_gelu_deco=use_gelu,num_heads_deco=num_enco_head, 
                att_dropout_deco=0,fourrier_feature_type="base2",num_fourier_feature_deco=3,depth_deco=3,
                use_pi=True,log_sampling=True,include_input=True,same_self_block=True)
deco.to(device)
deco.to(torch.float32)
optimizer=torch.optim.Adam(list(enco.parameters())+list( deco.parameters() ),lr=0.001  )


for bath in train_loader:
    x=bath[0]
    u=bath[1]
    x=x.to(device)
    u=u.to(device)
    target=u
    
    print("x shape before",x.shape)
    print("u shape before",u.shape)
    batch_size_tr,time_step,space_step,dim=x.shape #u.shape
    x=rearrange(x, "b T N d -> (b T) N d")
    u=rearrange(u, "b T N d -> (b T) N d")
    print("x shape after",x.shape)
    print("u shape after",u.shape)


    """sample,means,log_var,log_z_x=enco(x,u)
    print("sample=",sample.shape)
    print("means=",means.shape)
    print("logvar=",log_var.shape)
    print("---------------- DECODER -------------------------")
    out=deco(sample,x)
    out=rearrange(out,"(b T) N d -> b T N d",b=batch_size_tr)
    
    print(f"decoder output shape=={out.shape}")
    print("log_z_x=",log_z_x.shape)
    
    loss=loss_model(means,log_var,target,out,beta)
    loss.backward()
    print(f"losss={loss.item()}")"""
    break
    


"""sample,means,log_var,log_z_x=enco(x,u)
print("log_z_x=",log_z_x.shape)
print("sample=",sample.shape)
print(f"The encoder has {count_parameter(enco)} true requires_grad parameters")

x_deco=torch.rand(batch_size,N,x_dim).to(device)
#Z=torch.randn(batch_size,M,h).to(device)
out=deco(sample,x_deco)
print(f"The decoder has {count_parameter(deco)} true requires_grad parameters")
print(f"The total number of parameters are {count_parameter(enco)+count_parameter(deco)}")
print(f"decoder output shape=={out.shape}")"""


### Verification RC

In [1]:
import the_well
import inspect
import os
#print(os.path.dirname(the_well.__file__))

from the_well.data import WellDataModule

print(os.listdir(os.path.dirname(the_well.__file__)))

dir(the_well)

['benchmark', '__init__.py', 'utils', '__pycache__', 'data']


['__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 'data',
 'utils']

In [2]:
from the_well.data import WellDataModule

# The following line may take a couple of minutes to instantiate the datamodule
datamodule = WellDataModule(
    "hf://datasets/polymathic-ai/",
    "rayleigh_benard",
    batch_size=1,
)


In [3]:
test_dataloader = datamodule.test_dataloader()

In [5]:
for batch in test_dataloader:
    print(len(batch))
    print(type(batch))
    break

/opt/anaconda3/envs/deepdacPlusAroma/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/opt/anaconda3/envs/deepdacPlusAroma/lib/python3.11/site-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/deepdacPlusAroma/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/deepdacPlusAroma/lib/python3.11/site-packages/torch/utils/data/_utils/collate.py", line 401, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/deepdacPlusAroma/lib/python3.11/site-packages/torch/utils/data/_utils/collate.py", line 171, in collate
    {
  File "/opt/anaconda3/envs/deepdacPlusAroma/lib/pyth

## Affichage

In [ ]:
print(data_tr.shape)

### Solution simple : plots à différents instants (pour une publication)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Supposons vos données
# solution.shape = (nt, nx)
# nt = nombre de pas de temps
# nx = nombre de points spatiaux

# Création des axes
batch=4
nx=data_tr.shape[2]
nt=data_tr.shape[1]
solution=data_tr[batch]
x = np.linspace(0, 16, nx)
t = np.linspace(0, 4, nt,endpoint=True)

# Plot à différents instants
plt.figure(figsize=(12, 6))

# Sélectionner quelques instants
time_indices = [0, nt//4, nt//2, 3*nt//4, nt-1]
colors = plt.cm.viridis(np.linspace(0, 1, len(time_indices)))

for idx, color in zip(time_indices, colors):
    plt.plot(x, solution[idx, :], 
             color=color, 
             label=f't = {t[idx]:.1f}',
             linewidth=2)

plt.xlabel('x', fontsize=12)
plt.ylabel('u(x,t)', fontsize=12)
plt.title('Solution de l\'équation de Burgers', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

###  Diagramme spatio-temporel (heatmap) 

In [ ]:
batch=4
nx=data_tr.shape[2]
nt=data_tr.shape[1]
solution=data_tr[batch]
x = np.linspace(0, 16, nx)
t = np.linspace(0, 4, nt,endpoint=True)

plt.figure(figsize=(12, 8))
# 6. Énergie (norme L2)
ax6 = plt.subplot(2, 3, 6)
energy = np.sum(solution, axis=1) * (x[1] - x[0])
ax6.plot(t, energy, 'r-', linewidth=2)
ax6.set_xlabel('t')
ax6.set_ylabel('masse')
ax6.set_title('Massetotale')
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Animation

In [ ]:
batch=4
nx=data_tr.shape[2]
nt=data_tr.shape[1]
solution=data_tr[batch]
x = np.linspace(0, 16, nx)
t = np.linspace(0, 4, nt,endpoint=True)

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(10, 6))

line, = ax.plot(x, solution[0, :], 'b-', linewidth=2)
ax.set_xlim(x.min(),x.max())
ax.set_ylim(solution.min(), solution.max())
ax.set_xlabel('x')
ax.set_ylabel('u(x,t)')
ax.grid(True, alpha=0.3)
ax.set_title('Équation de Burgers - t = 0.0')

time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes)

def animate(frame):
    line.set_ydata(solution[frame, :])
    time_text.set_text(f't = {t[frame]:.2f}')
    ax.set_title(f'Équation de Burgers - t = {t[frame]:.2f}')
    return line, time_text

anim = FuncAnimation(fig, animate, frames=nt, interval=50, blit=True)

# Pour afficher dans Jupyter
plt.close()
HTML(anim.to_jshtml())

# Pour sauvegarder
anim.save('burgers_evolution.gif', writer='pillow', fps=20)

### Cache

In [ ]:
from functools import wraps
def cache_fn(f):
    cache = None

    @wraps(f)
    def cached_fn(*args, _cache=True, **kwargs):
        if not _cache:
            return f(*args, **kwargs)
        nonlocal cache
        if cache is not None:
            return cache
        cache = f(*args, **kwargs)
        return cache

    return cached_fn

In [ ]:
class NeatworkExample(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.dim=dim
        self.linear=nn.Linear(dim,dim)
    
    def forward(self,x):
        return x+self.linear(x)

x=torch.rand(1,3)
cached_slow = cache_fn( NeatworkExample(3) )

class NetExample(nn.ModuleList):
    def __init__(self,dim):
        super().__init__()
        cached_slow =cache_fn( lambda: NeatworkExample(dim) )
        self.mlp=nn.ModuleList([cached_slow(),cached_slow(),cached_slow()])

    def forward(self,x):
        x0=x
        i=0
        for l in self.mlp:
            print(f"i={i} result={l(x0)}")
            x=l(x)
        return x
dim=3
ex_net=NetExample(3)

In [ ]:
cached_slow(x) 

In [ ]:
for name,para in ex_net.named_parameters():
    print(name)

In [ ]:
y=ex_net(x)
print("y",y)

In [ ]:
criterion=nn.MSELoss()
optimizer=torch.optim.Adam(ex_net.parameters())
optimizer.zero_grad()
y=ex_net(x)
print("y=",y)
print("*****************************")
loss=criterion(y,torch.zeros(1,dim))
loss.backward()
optimizer.step()
y=ex_net(x)
print(y)


### Autre

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp=nn.Sequential(
            nn.Linear(1,10),
            nn.ReLU(),
            nn.Linear(10,1)
        )

    def forward(self,x):
        return self.mlp(x)

mymodel=MyModel()
mymodel2=MyModel()
criterion=torch.nn.MSELoss()
optimizer=torch.optim.Adam(mymodel.parameters() ,lr=0.001 )
X=torch.randn(10,5,1)
Y=torch.randn(10,5,1)

for name,para in list(mymodel.named_parameters())+ list(mymodel2.named_parameters()):
    print(name,para)


"""for _ in range(1):
    mymodel.train()
    optimizer.zero_grad()
    out=mymodel(X)
    loss=criterion(Y,out)
    loss.backward()
    optimizer.step()

# Verifier nan dans les parametre 
for name, param in mymodel.named_parameters():
    if torch.isnan(param).any():
        print("NaN dans les poids :", name)

torch.save(mymodel,checkpoint_nan)



with checkpoint_nan.open("rb") as fp:
    mymodel2=torch.load(fp, weights_only=False,map_location=torch.device('cpu')) #on recommence depui s l e modele sauvegarde
    

# verifier nan dans grad des parametres
for name, param in mymodel2.named_parameters():
    if param.grad is not None:
        if torch.isnan(param.grad).any():
            print("NaN dans le gradient :", name)
        
        else:
            print(name, param.grad)
    
    else:
        print(f"{name} has None gradient")
"""

In [ ]:
from DiT import *
dit=DiT(input_size=16,
        num_tokens=16)
x=torch.randn(32,16*2,16)
t=torch.linspace(0,1,32)
y=dit(x,t)
print(y.shape)

In [ ]:
import numpy as np
AA=np.array([1,2,3,4])
AA.tolist()

In [ ]:
import torch 

try:
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
except Exception as e:
    # Fallback to CPU if device selection fails
    print(f"Warning: Device selection failed ({e}), using CPU")
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
A=torch.ones(2,3)
B=A.unsqueeze(1)
B.shape

In [ ]:
A=torch.ones(2,3).unsqueeze(-1)
B=2*torch.tensor([0,1.,2,3,4]).view(1,-1)
C=torch.matmul(A,B)
C